<a href="https://colab.research.google.com/github/Somendar/dataviz-excercises-SomendarKumarDas/blob/main/lecture08_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [2]:
import plotly.express as px
import pandas as pd

# Gapminder data
df = px.data.gapminder()
df_2007 = df[df["year"] == 2007]

global_avg = df_2007["lifeExp"].mean()

df_2007["life_dev"] = df_2007["lifeExp"] - global_avg

# choropleth
fig = px.choropleth(
    df_2007,
    locations="iso_alpha",
    color="life_dev",
    hover_name="country",
    hover_data={
        "lifeExp":":.2f",
        "life_dev":":.2f",
        "gdpPercap":":,.0f",
        "pop":":,.0f"
    },
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title=f"Deviation of Country Life Expectancy from Global Mean ({global_avg:.2f} years)"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

/tmp/ipykernel_9234/2662292827.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2007["life_dev"] = df_2007["lifeExp"] - global_avg


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** *(URL or description)*

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

*Your answer here*

**Colour scale chosen** (sequential or diverging) **and reason:**

*Your answer here*


In [3]:
import pandas as pd
import plotly.express as px
import json
import numpy as np

# Load GeoJSON
geojson_url = "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_110m_admin_1_states_provinces_shp.geojson"

geojson = pd.read_json(geojson_url)

# Read using requests
import requests

response = requests.get(geojson_url)
geojson_data = response.json()

# Extract province names
regions = []

for feature in geojson_data["features"]:
    region_name = feature["properties"]["name"]
    regions.append(region_name)

# Create dataframe
region_df = pd.DataFrame({
    "region": regions
})

np.random.seed(42)

region_df["value"] = np.random.randint(
    10,
    100,
    len(region_df)
)

# Choropleth
fig = px.choropleth(
    region_df,
    geojson=geojson_data,
    locations="region",
    featureidkey="properties.name",
    color="value",
    hover_name="region",
    color_continuous_scale="Viridis",
    title="Sample Metric Across States/Provinces"
)

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    title_x=0.5
)

fig.show()